# Demo 3：公平性、证据交付、异常与模型推断

本 Notebook 使用一个带敏感字段的合成二分类表格，集中讲解 API 中容易被忽略的能力：`sensitive_features`、公平性结果、证据验收、`retry_evidence()`、异常捕获和两个持久化模型文件。

In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from tabularx import AutoEstimator
from tabularx.exceptions import DataError, SplitError, TabularXError


In [2]:
features, y = make_classification(
    n_samples=240, n_features=6, n_informative=4,
    n_redundant=0, random_state=42, weights=[0.55, 0.45]
)
X = pd.DataFrame(features, columns=[f"feature_{i}" for i in range(6)])
X["group"] = np.where(np.arange(len(X)) % 2 == 0, "A", "B")
y = pd.Series(y, name="target")
print(X.head())
print(y.value_counts())


   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5 group
0  -2.943494  -0.892953  -2.824271   0.554803  -0.024105  -0.579701     A
1  -2.314469  -1.543272  -1.110622   0.860481  -0.093765   0.016885     B
2   0.849346   0.064323   1.814302   2.120802   2.042985  -0.045881     A
3   5.629101  -3.405797  -0.369532  -2.931338   0.383612  -0.020019     B
4   0.339128  -2.116446  -0.976066  -1.016300  -1.081420  -0.462949     A
target
0    132
1    108
Name: count, dtype: int64


## 1. 请求公平性评估

将原始表格中的敏感列名传给 `fit(..., sensitive_features=[...])`。当前公平性指标主要适用于 0/1 二分类；公平性结果用于审计，不等同于公平性修正。

In [3]:
ae = AutoEstimator(task="classification", output_dir="./outputs/demo3")
ae.param.model_search.models = ["random_forest"]
ae.param.model_search.n_trials = 2
ae.param.model_search.n_jobs = 1
ae.param.shap.max_explain_samples = 24
ae.param.fairness.sensitive_features = ["group"]
ae.fit(X, y, sensitive_features=["group"])


[I 2026-10-07 22:23:51,696] A new study created in memory with name: random_forest_binary


<AutoEstimator task='binary' output_dir=./outputs/demo3 已完成>

In [4]:
r = ae.result
print("公平性结果：", r.fairness)
if r.fairness is not None:
    for item in r.fairness.feature_results:
        print(item)


公平性结果： FairnessResult(enabled=True, sensitive_features=['group'], feature_results=[FairnessFeatureResult(feature_name='group', is_dropped=False, dropped_reason=None, groups=[FairnessGroupResult(group_name='B', n_samples=21, positive_rate=0.3333333333333333, accuracy=0.8571428571428571, tpr=0.75, fpr=0.07692307692307693, support=21), FairnessGroupResult(group_name='A', n_samples=27, positive_rate=0.48148148148148145, accuracy=0.8888888888888888, tpr=0.8571428571428571, fpr=0.07692307692307693, support=27)], demographic_parity_diff=0.14814814814814814, equalized_odds_tpr_diff=0.1071428571428571, equalized_odds_fpr_diff=0.0, dp_violation=True, eo_tpr_violation=True, eo_fpr_violation=False)], error=None)
FairnessFeatureResult(feature_name='group', is_dropped=False, dropped_reason=None, groups=[FairnessGroupResult(group_name='B', n_samples=21, positive_rate=0.3333333333333333, accuracy=0.8571428571428571, tpr=0.75, fpr=0.07692307692307693, support=21), FairnessGroupResult(group_name='A', n_

## 2. 读取证据交付状态

`EvidenceDelivery.audit()` 返回每个交付项的状态。常见状态包括 `generated`、`failed`、`missing`、`not_enabled` 和 `not_applicable`。报告存在不代表所有上游证据都完整。

In [5]:
delivery = ae.evidence_.audit()
print("是否完整：", delivery["complete"])
for record in delivery["records"]:
    print(record)


是否完整： True
selection_lock
model
reload_check
shap
fairness
model_scores
audit
model_search_audit
diagnostics/random_forest
trials/random_forest
profile_report
split_report
preprocess_report
search_report
model_report/random_forest
explanation_report
full_report
pdf_report


## 3. 局部补做失败证据

`retry_evidence()` 必须在原来的 Python 会话中调用。它复用已经拟合的模型、数据和参数，不会重新训练；也可以传入键名列表，只重试指定的可恢复项。

下面的调用在没有失败项时会保持当前状态不变。

In [6]:
status = ae.retry_evidence()
print(status)
# 也可以只重试某个项目：
# status = ae.retry_evidence(["shap"])


{'schema_version': '1.0', 'run_id': 'afa5f7f1ec3643d89fc80a68180a164d', 'task': 'binary', 'complete': True, 'required_count': 17, 'generated_required_count': 17, 'completeness': 1.0, 'boundary': 'Delivery acceptance only; not scientific validity or reproducibility', 'records': {'selection_lock': {'required': True, 'applicable': True, 'condition': 'Declared for task=binary', 'enabled': True, 'status': 'generated', 'reason': 'Producer completed; declared files present and nonempty', 'paths': ['evidence/selection_lock.json'], 'attempts': [{'started_at': '2026-10-07T14:23:53.477685+00:00', 'ended_at': '2026-10-07T14:23:53.478683+00:00', 'status': 'generated', 'reason': 'Producer completed; declared files present and nonempty', 'paths': ['evidence/selection_lock.json']}], 'recoverable': True}, 'model': {'required': True, 'applicable': True, 'condition': 'Declared for task=binary', 'enabled': True, 'status': 'generated', 'reason': 'Producer completed; declared files present and nonempty', 'p

## 4. 读取报告路径和结构化证据

`result.report_paths` 用于程序化获取 HTML 报告；`delivery.json`、`audit.json`、`selection_lock.json` 和 `reload_check.json` 是运行目录中的机器可读证据。

In [7]:
for name, path in r.report_paths.items():
    print(name, path)
print("运行目录：", ae.output_dir)


profile C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo3\runs\afa5f7f1ec3643d89fc80a68180a164d\reports\profile.html
split C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo3\runs\afa5f7f1ec3643d89fc80a68180a164d\reports\split.html
preprocess C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo3\runs\afa5f7f1ec3643d89fc80a68180a164d\reports\preprocess.html
model_search C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo3\runs\afa5f7f1ec3643d89fc80a68180a164d\reports\model_search.html
explanation C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo3\runs\afa5f7f1ec3643d89fc80a68180a164d\reports\explanation.html
full C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo3\runs\afa5f7f1ec3643d89fc80a68180a164d\report.html
evidence C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo3\runs\afa5f7f1ec3643d89fc80a68180a164d\evidence\delivery.html
evidence_json C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo3\runs\afa5f7f1ec3643d89fc80a68180a164d\evid

## 5. `pipeline.joblib` 与 `best_model.joblib`

`pipeline.joblib` 包含预处理器和模型，推荐用于接收原始 DataFrame。`best_model.joblib` 只包含底层模型，必须输入预处理后的矩阵。系统会在交付阶段执行重载一致性核对。

In [8]:
pipeline = joblib.load(ae.output_dir / "pipeline.joblib")
raw_predictions = pipeline.predict(X.head(5))
print("原始表格推断：", raw_predictions)

best_model = joblib.load(ae.output_dir / "best_model.joblib")
processed = r.preprocessed.X_test.head(5)
print("预处理矩阵推断：", best_model.predict(processed))


原始表格推断： [0 0 1 0 1]
预处理矩阵推断： [0 1 0 1 1]


## 6. 捕获 TabularX 异常

输入数据问题通常属于 `DataError`，划分问题属于 `SplitError`；无法预先确定阶段时可以捕获 `TabularXError`。异常对象包含阶段信息，便于定位问题。

In [9]:
try:
    bad = AutoEstimator(task="classification", output_dir="./outputs/demo3-invalid")
    bad.fit(X.iloc[:-1], y)  # 故意制造 X 与 y 长度不一致
except SplitError as exc:
    print("划分阶段错误：", exc)
except DataError as exc:
    print("数据错误：", exc)
except TabularXError as exc:
    print("TabularX 错误：", exc)


数据错误： [profile] X 行数（239）与 y 长度（240）不一致。请确保 X 与 y 对应同一批样本。


## 本例小结与边界

本例覆盖公平性评估、证据验收、局部补做、报告路径、模型持久化和异常体系。

- 交付完整不代表模型准确、公平或审计无风险。
- 局部补做只支持原 Python 会话，进程退出后不能仅凭 JSON 自动恢复。
- 重载核对验证兼容环境中的预测一致性，不代表跨操作系统、依赖版本或硬件完全复现。
- 生产环境只应加载可信来源的 joblib 文件。